# i01 · ¿Cuánto ha crecido la población de España y quién explica ese crecimiento?

- **Post:** `posts/i01-population-nationality/index.qmd`
- **Afirmación:** «España solo crece gracias a la inmigración.» (y la subafirmación: «han llegado 6 millones de inmigrantes durante el gobierno de Sánchez»)
- **Alcance:** solo nivel **nacional**. El análisis territorial (CCAA, provincias) queda para un artículo futuro; las funciones de abajo se reutilizan cambiando la columna `grupo` por `territorio`.
- **Datos:** INE, Estadística Continua de Población, tabla 56938 (`data/raw/ine_poblacion/56938.csv`). Se usa solo la población a **1 de enero** (2002–2025).
  - descarga: `python src/download/download_ine_poblacion.py`
- **Preguntas:**
  - **Q1.** ¿Cómo ha crecido la población residente? ¿A qué ritmos y en qué periodos?
  - **Q2.** ¿Cómo se compone ese crecimiento según nacionalidad y país de nacimiento?
  - **Q3.** ¿Cuánto del cambio de población se explica por el saldo natural y cuánto por el saldo migratorio exterior? *(pendiente de datos)*
  - **Q4.** ¿Cuántos llegan, cuántos salen y cuántos se quedan? ¿Cómo se relacionan los flujos con el aumento del stock de extranjeros? *(pendiente de datos)*

## 0. Criterios pre-registrados

*Escritos antes de mirar los resultados. Cualquier cambio posterior se anota en el registro de cambios del final.*

**Tesis.** **A**: la inmigración explica la **mayor parte** del crecimiento de la población. **B**: no la explica.

**Medida (Q2).** Contribución de un grupo = variación de su población en el periodo ÷ variación de la población total (%). Las poblaciones son a 1 de enero; el periodo [ini, fin] es `stock(fin) − stock(ini)`, de modo que «2002–2008» recoge los años naturales 2002–2007.

| Contribución del grupo extranjero | Veredicto |
|---|---|
| ≥ 75 % | **Apunta a A** |
| 25 % – 75 % | **Mixta** |
| < 25 % | **Apunta a B** |

- **Versión literal («solo»):** se cumple si la contribución es ≥ 95 %. Se informa siempre como etiqueta aparte.
- **No aplicable:** variación total ≤ 0 o < 0,5 % de la población inicial. **Insuficiente:** los datos no permiten calcularlo.
- **Confianza:** *Alta* por defecto; *Media* si la contribución está a ≤ 5 puntos de un umbral o si las dos definiciones (Q2) discrepan; *Baja* si además hay un problema de calidad de datos señalado en el cuaderno.
- **Periodos:** 2002–2008, 2008–2014, 2014–2019, 2019–2025 y total 2002–2025. Sensibilidad: bloques de 5 años.
- **Veredicto global:** A o B si se cumple en el total **y** en ≥ 3 de los 4 periodos; si no, **Mixta**.
- **Q2 con dos definiciones** (nacionalidad y país de nacimiento): deben coincidir; si no, **Mixta** (confianza ≤ Media).
- **Redacción:** «Apunta a A: la inmigración explica el X %. La versión literal («solo») [se cumple / no se cumple].»

**Q3 (saldo natural vs. migratorio).** Contribución del saldo migratorio exterior = saldo migratorio ÷ ΔPoblación (puede superar 100 %). Se aplican los mismos umbrales A/B/Mixta. El **residuo** = ΔPoblación − (saldo natural + saldo migratorio) se muestra siempre; si |residuo| > 20 % de ΔPoblación → **Insuficiente**. Cautela: los nacidos de madre extranjera computan en el saldo natural.

**Q4 («6 millones»).** Medidas: llegadas brutas (todas / extranjeros), saldo neto (todos / extranjeros) y variación del stock de nacidos en el extranjero. Dos inicios: 2018 y 2020. **Respaldada** si el saldo neto está entre 5,4 y 6,6 M; **Depende de la medida** si solo las llegadas brutas alcanzan esa cifra; **No respaldada** si ninguna medida llega a 5,4 M. Coherencia flujo–stock: se señala si |Δstock − saldo de extranjeros| > 25 % de Δstock.

## 1. Preparación

In [1]:
import sys
from pathlib import Path
import polars as pl
import plotly.io as pio
from plotly.subplots import make_subplots

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "_quarto.yml").exists())
sys.path.insert(0, str(ROOT))
from src.utils.plotting import ACCENT, GREY, PALETTE, init_plotly, new_figure
init_plotly()
pio.templates["mirada"].layout.paper_bgcolor = "white"   # fondo blanco para trabajar en el notebook
pio.templates["mirada"].layout.plot_bgcolor = "white"

RAW = ROOT / "data" / "raw" / "ine_poblacion"
LIMITES = [2002, 2008, 2014, 2019, 2025]            # periodos principales (stock a 1 de enero)
LIM_SENS = [2002, 2007, 2012, 2017, 2022, 2025]     # sensibilidad: bloques de 5 años
LIM_GOB = {"Sánchez desde 2018": [2002, 2004, 2012, 2018, 2025],
           "Sánchez desde 2020": [2002, 2004, 2012, 2020, 2025]}

## 2. Funciones

Todas las funciones del análisis, juntas. Esperan una tabla `anual` con columnas `año`, `grupo`, `poblacion` (stock a 1 de enero).

In [2]:
MESES = {"enero": "01", "febrero": "02", "marzo": "03", "abril": "04", "mayo": "05", "junio": "06",
         "julio": "07", "agosto": "08", "septiembre": "09", "octubre": "10", "noviembre": "11", "diciembre": "12"}

MAPS = {  # "nacionalidad | país de nacimiento" -> grupo
    "nacionalidad": {"Española | Total": "Española", "Extranjera | Total": "Extranjera"},
    "nacimiento": {"Total | España": "Nacidos en España", "Total | Extranjero": "Nacidos en el extranjero"},
    "cuatro": {"Española | España": "Españoles nacidos en España",
               "Española | Extranjero": "Españoles nacidos en el extranjero",
               "Extranjera | España": "Extranjeros nacidos en España",
               "Extranjera | Extranjero": "Extranjeros nacidos en el extranjero"},
}
COL_NAC = {"Española": GREY, "Extranjera": ACCENT}
COL_NACIM = {"Nacidos en España": GREY, "Nacidos en el extranjero": ACCENT}
COL_4 = dict(zip(MAPS["cuatro"].values(), [GREY, PALETTE[1], PALETTE[2], ACCENT]))
COL_TOTAL = {"Total": ACCENT}


def es(x, d=0):
    # formato español: 1.234.567,89
    return f"{x:,.{d}f}".replace(",", "X").replace(".", ",").replace("X", ".")


# ---------- datos ----------
def cargar_ecp(path):
    # Tabla 56938: totales de sexo y edad, solo 1 de enero. Una fila por año × nacionalidad × país de nacimiento.
    return (pl.scan_csv(path, separator=";")
        .filter(pl.col("Sexo") == "Total", pl.col("Edad simple") == "Todas las edades")
        .with_columns(
            fecha=pl.col("Periodo").str.replace_many(MESES).str.to_date("%d de %m de %Y"),
            poblacion=pl.col("Total").str.replace_all(r"\.", "").str.replace(",", ".")
                        .cast(pl.Float64, strict=False).round().cast(pl.Int64))
        .filter(pl.col("poblacion").is_not_null(), pl.col("fecha").dt.month() == 1, pl.col("fecha").dt.day() == 1)
        .select(año=pl.col("fecha").dt.year(), nacionalidad="Nacionalidad", pais="País de nacimiento", poblacion="poblacion")
        .collect())


def grupos(base, por):
    # por: "nacionalidad" | "nacimiento" | "cuatro"
    m = MAPS[por]
    return (base.with_columns(grupo=pl.concat_str(["nacionalidad", "pais"], separator=" | "))
        .filter(pl.col("grupo").is_in(list(m)))
        .with_columns(pl.col("grupo").replace_strict(m))
        .select("año", "grupo", "poblacion").sort("grupo", "año"))


def comprobar(anual, total):
    # años por grupo + diferencia máxima entre la suma de grupos y el total
    desc = anual.group_by("grupo").agg(n=pl.len(), desde=pl.col("año").min(), hasta=pl.col("año").max())
    dif = (anual.group_by("año").agg(suma=pl.col("poblacion").sum()).join(total, on="año")
        .select(dif_max=(pl.col("suma") - pl.col("poblacion")).abs().max()))
    return desc, dif


# ---------- cálculo ----------
def variacion_anual(anual):
    # variación entre 1 de enero t-1 y 1 de enero t, dentro de cada grupo
    return anual.sort("grupo", "año").with_columns(
        var_abs=pl.col("poblacion").diff().over("grupo"),
        var_pct=100 * pl.col("poblacion").pct_change().over("grupo"))


def resumen_periodos(anual, limites, con_total=True):
    # un periodo [ini, fin] = stock(fin) - stock(ini); con_total añade el periodo completo
    pares = list(zip(limites[:-1], limites[1:])) + ([(limites[0], limites[-1])] if con_total else [])
    tipos = ["periodo"] * (len(limites) - 1) + (["total"] if con_total else [])
    periodos = pl.DataFrame({"ini": [p[0] for p in pares], "fin": [p[1] for p in pares], "tipo": tipos})
    ini = anual.rename({"año": "ini", "poblacion": "pob_ini"})
    fin = anual.rename({"año": "fin", "poblacion": "pob_fin"})
    return (periodos.join(ini, on="ini").join(fin, on=["fin", "grupo"])
        .with_columns(años=pl.col("fin") - pl.col("ini"), var_abs=pl.col("pob_fin") - pl.col("pob_ini"))
        .with_columns(
            var_pct_total=100 * (pl.col("pob_fin") / pl.col("pob_ini") - 1),
            media_anual=pl.col("var_abs") / pl.col("años"),
            tasa_anual=100 * ((pl.col("pob_fin") / pl.col("pob_ini")) ** (1 / pl.col("años")) - 1),
            contribucion=100 * pl.col("var_abs") / pl.col("var_abs").sum().over("ini", "fin"),
            periodo=pl.format("{}–{}", pl.col("ini"), pl.col("fin")))
        .sort("tipo", "ini", "grupo"))


def tabla(res):
    cols = ["periodo", "grupo", "pob_ini", "pob_fin", "var_abs", "var_pct_total", "media_anual", "tasa_anual", "contribucion"]
    return res.select([c for c in cols if c in res.columns]).with_columns(pl.col(pl.Float64).round(2))


# ---------- veredicto (criterios de la sección 0) ----------
def clasificar(contrib, var_total, pob_ini, a=75, b=25):
    if var_total <= 0 or var_total < 0.005 * pob_ini:
        return "No aplicable", None
    v = "Apunta a A" if contrib >= a else "Apunta a B" if contrib < b else "Mixta"
    return v, "Media" if min(abs(contrib - a), abs(contrib - b)) <= 5 else "Alta"


def tabla_veredictos(res, ext):
    # una fila por periodo para el grupo `ext` (el grupo extranjero de la definición)
    filas = []
    for r in res.filter(pl.col("grupo") == ext).iter_rows(named=True):
        t = res.filter(pl.col("ini") == r["ini"], pl.col("fin") == r["fin"])
        v, conf = clasificar(r["contribucion"], t["var_abs"].sum(), t["pob_ini"].sum())
        filas.append(dict(periodo=r["periodo"], tipo=r["tipo"], contribucion=round(r["contribucion"], 1),
                          literal=r["contribucion"] >= 95, veredicto=v, confianza=conf))
    return pl.DataFrame(filas)


def veredicto_global(t):
    tot = t.filter(pl.col("tipo") == "total")["veredicto"][0]
    n = t.filter(pl.col("tipo") == "periodo", pl.col("veredicto") == tot).height
    return tot if tot in ("Apunta a A", "Apunta a B") and n >= 3 else "Mixta"


def frase(t, global_):
    r = t.filter(pl.col("tipo") == "total").row(0, named=True)
    lit = "se cumple" if r["literal"] else "no se cumple"
    return f"{global_}: la inmigración explica el {es(r['contribucion'], 1)} %. La versión literal («solo») {lit}."


# ---------- gráficos ----------
def grafico_anual(anual, total, titulo, colores, limites=None):
    # barras = variación anual por grupo (en el año del stock de llegada), línea = población total (eje dcho.)
    v = variacion_anual(anual).drop_nulls("var_abs").with_columns(ant=pl.col("año") - 1)
    fig = make_subplots(specs=[[{"secondary_y": True}]])
    for g in v["grupo"].unique(maintain_order=True):
        d = v.filter(pl.col("grupo") == g)
        fig.add_bar(x=d["año"], y=d["var_abs"], name=g, marker_color=colores[g], customdata=d["ant"].to_list(),
                    hovertemplate="%{customdata} → %{x}: %{y:,.0f}<extra>" + g + "</extra>")
    if v["grupo"].n_unique() > 1:
        t = v.group_by("año").agg(pl.col("var_abs").sum()).sort("año")
        fig.add_scatter(x=t["año"], y=t["var_abs"], name="Variación total", mode="markers",
                        marker=dict(symbol="diamond", size=9, color="black"))
    fig.add_scatter(x=total["año"], y=total["poblacion"], name="Población total (eje dcho.)", mode="lines",
                    line=dict(color="#111827", width=2), secondary_y=True)
    if limites:
        res = resumen_periodos(total.with_columns(grupo=pl.lit("Total")), limites, con_total=False)
        for i, r in enumerate(res.iter_rows(named=True)):
            x0, x1 = r["ini"] + 0.5, r["fin"] + 0.5
            fig.add_vrect(x0=x0, x1=x1, fillcolor=GREY, opacity=0.10 if i % 2 == 0 else 0, line_width=0, layer="below")
            fig.add_scatter(x=[x0, x1], y=[r["media_anual"]] * 2, mode="lines", showlegend=False,
                            line=dict(color="#0d9488", dash="dash", width=2), hoverinfo="skip")
            fig.add_annotation(x=(x0 + x1) / 2, y=1, yref="paper", yanchor="bottom", showarrow=False,
                               text=f"{es(r['tasa_anual'], 2)} %/año<br>{es(r['media_anual'])} pers./año", font=dict(size=11))
    fig.update_layout(title=dict(text=titulo, x=0), barmode="relative", height=500, margin=dict(t=110, b=110, r=70),
                      legend=dict(orientation="h", y=-0.12, yanchor="top"))
    fig.update_yaxes(title_text="Variación anual (personas)", secondary_y=False)
    fig.update_yaxes(title_text="Población a 1 de enero", showgrid=False, secondary_y=True)
    return fig


def grafico_periodos(res, colores, titulo):
    # media anual por periodo y grupo; el texto de cada barra es la contribución al crecimiento total
    r = res.filter(pl.col("tipo") == "periodo")
    fig = new_figure("Variación media anual (personas)", height=420)
    for g in r["grupo"].unique(maintain_order=True):
        d = r.filter(pl.col("grupo") == g)
        fig.add_bar(x=d["periodo"], y=d["media_anual"], name=g, marker_color=colores[g],
                    text=[f"{es(c)} %" for c in d["contribucion"]], textposition="inside")
    t = r.group_by("periodo", maintain_order=True).agg(pl.col("media_anual").sum())
    fig.add_scatter(x=t["periodo"], y=t["media_anual"], name="Total", mode="markers",
                    marker=dict(symbol="diamond", size=11, color="black"))
    fig.update_layout(title=dict(text=titulo, x=0), barmode="relative")
    return fig


def grafico_cuota(anual, total, colores, titulo, excluir=()):
    # peso de cada grupo en la población total (se puede excluir el grupo dominante)
    c = (anual.join(total.rename({"poblacion": "total"}), on="año")
        .filter(~pl.col("grupo").is_in(list(excluir)))
        .with_columns(cuota=100 * pl.col("poblacion") / pl.col("total")).sort("grupo", "año"))
    fig = new_figure("% de la población total", height=420)
    for g in c["grupo"].unique(maintain_order=True):
        d = c.filter(pl.col("grupo") == g)
        fig.add_scatter(x=d["año"], y=d["cuota"], name=g, mode="lines", line=dict(color=colores[g], width=3),
                        hovertemplate="%{y:.1f} %<extra>" + g + "</extra>")
        fig.add_annotation(x=d["año"][-1], y=d["cuota"][-1], text=f"{es(d['cuota'][-1], 1)} %", showarrow=False,
                           xanchor="left", xshift=6, font=dict(color=colores[g]))
    fig.update_layout(title=dict(text=titulo, x=0), margin=dict(r=70))
    fig.update_yaxes(ticksuffix=" %", tickformat=".0f")
    return fig

## 3. Datos

Una sola tabla: población residente a 1 de enero por nacionalidad × país de nacimiento (sin desglose por sexo ni edad).

In [3]:
base = cargar_ecp(RAW / "56938.csv")
total = base.filter(pl.col("nacionalidad") == "Total", pl.col("pais") == "Total").select("año", "poblacion").sort("año")
anual_total = total.with_columns(grupo=pl.lit("Total"))
print(base.shape, "· años:", total["año"].min(), "–", total["año"].max(), "·", total.height, "stocks")
total.tail(3)

(3168, 4) · años: 2002 – 2025 · 24 stocks


año,poblacion
i32,i64
2023,48085361
2024,48619695
2025,49128297


## 4. Q1 · ¿Cómo ha crecido la población residente? ¿A qué ritmos y en qué periodos?

### 4.1 Cifra titular (2002 → 2025)

In [4]:
tabla(resumen_periodos(anual_total, [LIMITES[0], LIMITES[-1]], con_total=False))

periodo,grupo,pob_ini,pob_fin,var_abs,var_pct_total,media_anual,tasa_anual,contribucion
str,str,i64,i64,i64,f64,f64,f64,f64
"""2002–2025""","""Total""",41035271,49128297,8093026,19.72,351870.7,0.79,100.0


### 4.2 Por periodos

In [5]:
tabla(resumen_periodos(anual_total, LIMITES))

periodo,grupo,pob_ini,pob_fin,var_abs,var_pct_total,media_anual,tasa_anual,contribucion
str,str,i64,i64,i64,f64,f64,f64,f64
"""2002–2008""","""Total""",41035271,45668938,4633667,11.29,772277.83,1.8,100.0
"""2008–2014""","""Total""",45668938,46495744,826806,1.81,137801.0,0.3,100.0
"""2014–2019""","""Total""",46495744,46918951,423207,0.91,84641.4,0.18,100.0
"""2019–2025""","""Total""",46918951,49128297,2209346,4.71,368224.33,0.77,100.0
"""2002–2025""","""Total""",41035271,49128297,8093026,19.72,351870.7,0.79,100.0


### 4.3 Sensibilidad a los límites (bloques de 5 años)

In [6]:
tabla(resumen_periodos(anual_total, LIM_SENS))

periodo,grupo,pob_ini,pob_fin,var_abs,var_pct_total,media_anual,tasa_anual,contribucion
str,str,i64,i64,i64,f64,f64,f64,f64
"""2002–2007""","""Total""",41035271,44784659,3749388,9.14,749877.6,1.76,100.0
"""2007–2012""","""Total""",44784659,46818216,2033557,4.54,406711.4,0.89,100.0
"""2012–2017""","""Total""",46818216,46497393,-320823,-0.69,-64164.6,-0.14,100.0
"""2017–2022""","""Total""",46497393,47486727,989334,2.13,197866.8,0.42,100.0
"""2022–2025""","""Total""",47486727,49128297,1641570,3.46,547190.0,1.14,100.0
"""2002–2025""","""Total""",41035271,49128297,8093026,19.72,351870.7,0.79,100.0


### 4.4 Variación año a año

In [7]:
variacion_anual(anual_total).select("año", "poblacion", "var_abs", pl.col("var_pct").round(2))

año,poblacion,var_abs,var_pct
i32,i64,i64,f64
2002,41035271,null,null
2003,41827836,792565,1.93
2004,42547454,719618,1.72
2005,43296335,748881,1.76
2006,44009969,713634,1.65
…,…,…,…
2021,47400798,82748,0.17
2022,47486727,85929,0.18
2023,48085361,598634,1.26


### 4.5 Gráfico

In [8]:
grafico_anual(anual_total, total, "Variación anual de la población residente y población total", COL_TOTAL, LIMITES).show()

**Lectura Q1** · *borrador de Claude para que lo revises y lo reescribas con tu voz*

- **Crecimiento total.** La población residente pasa de 41.035.271 (1-I-2002) a 49.128.297 (1-I-2025): +8.093.026 personas, +19,7 %, de media 351.871 al año (0,79 % anual compuesto).
- **Tres ritmos muy distintos.**
  - 2002–2008: +4.633.667 (772.278/año, 1,80 % anual). El año con mayor aumento es 2008 (+884.279).
  - 2008–2014 y 2014–2019: crecimiento casi nulo (137.801/año y 84.641/año; 0,30 % y 0,18 % anual). Entre 2013 y 2016 la población **baja** cuatro años seguidos (el mínimo, −216.906, en 2014).
  - 2019–2025: +2.209.346 (368.224/año, 0,77 % anual), un ritmo medio similar al de todo el periodo pero concentrado en 2019–2020 y 2023–2025.
- **Sensibilidad.** Con bloques de 5 años el patrón se mantiene (1,76 % → 0,89 % → −0,14 % → 0,42 % → 1,14 % anual), pero el cierre de cada periodo cambia las medias: por eso la tasa de un periodo no debe leerse sin su gráfico anual.
- **A comprobar.** En 2021 y 2022 el aumento es solo de +82.748 y +85.929, y en 2023 salta a +598.634. Puede ser un efecto real (pandemia, y luego repunte) o un artefacto de la serie (cambio de metodología/revisión en la ECP). Conviene contrastarlo con la nota metodológica del INE antes de interpretar ese tramo.
- Esto describe **cuánto** crece la población, no **por qué**; la composición está en Q2 y las causas (saldos natural y migratorio) en Q3.

## 5. Q2 · ¿Cómo se compone el crecimiento según nacionalidad y país de nacimiento?

### 5.1 Por nacionalidad

In [9]:
nac = grupos(base, "nacionalidad")
res_nac = resumen_periodos(nac, LIMITES)
display(*comprobar(nac, total))
tabla(res_nac)

grupo,n,desde,hasta
str,u32,i32,i32
"""Española""",24,2002,2025
"""Extranjera""",24,2002,2025


dif_max
i64
1


periodo,grupo,pob_ini,pob_fin,var_abs,var_pct_total,media_anual,tasa_anual,contribucion
str,str,i64,i64,i64,f64,f64,f64,f64
"""2002–2008""","""Española""",39297299,40582643,1285344,3.27,214224.0,0.54,27.74
"""2002–2008""","""Extranjera""",1737972,5086295,3348323,192.66,558053.83,19.6,72.26
"""2008–2014""","""Española""",40582643,41819392,1236749,3.05,206124.83,0.5,149.58
"""2008–2014""","""Extranjera""",5086295,4676352,-409943,-8.06,-68323.83,-1.39,-49.58
"""2014–2019""","""Española""",41819392,42068189,248797,0.59,49759.4,0.12,58.79
"""2014–2019""","""Extranjera""",4676352,4850762,174410,3.73,34882.0,0.74,41.21
"""2019–2025""","""Española""",42068189,42216326,148137,0.35,24689.5,0.06,6.71
"""2019–2025""","""Extranjera""",4850762,6911971,2061209,42.49,343534.83,6.08,93.29
"""2002–2025""","""Española""",39297299,42216326,2919027,7.43,126914.22,0.31,36.07


In [10]:
grafico_anual(nac, total, "Variación anual por nacionalidad", COL_NAC, LIMITES).show()
grafico_periodos(res_nac, COL_NAC, "Variación media anual por periodo y nacionalidad (% = contribución)").show()
grafico_cuota(nac, total, COL_NAC, "Peso de la población de nacionalidad extranjera", excluir=["Española"]).show()

**Lectura 5.1 · Nacionalidad** · *borrador*

- La población de nacionalidad extranjera pasa de 1.737.972 (4,2 % del total) a 6.911.971 (14,1 %): +5.173.999, el **63,9 %** del crecimiento total 2002–2025. La española crece +2.919.027 (36,1 %).
- Por periodos, la contribución extranjera es 72,3 % (2002–2008), **−49,6 %** (2008–2014), 41,2 % (2014–2019) y 93,3 % (2019–2025). En 2008–2014 la población extranjera **disminuye** (−409.943, −8,1 %) y todo el crecimiento procede de la española (+1.236.749).
- Según los criterios de la sección 0: 2002–2008 queda en *Mixta* con confianza Media (72,3 % está a menos de 5 puntos del umbral de 75 %); 2008–2014 apunta a B; 2014–2019 es *Mixta*; 2019–2025 apunta a A. El total (63,9 %) es *Mixta*.
- **Matiz.** «Española» incluye a quienes han adquirido la nacionalidad; esa transferencia resta población «extranjera» y suma «española» sin que nadie se mueva (ver 5.2 y 5.4).

### 5.2 Por país de nacimiento

In [11]:
nacim = grupos(base, "nacimiento")
res_nacim = resumen_periodos(nacim, LIMITES)
display(*comprobar(nacim, total))
tabla(res_nacim)

grupo,n,desde,hasta
str,u32,i32,i32
"""Nacidos en España""",24,2002,2025
"""Nacidos en el extranjero""",24,2002,2025


dif_max
i64
1


periodo,grupo,pob_ini,pob_fin,var_abs,var_pct_total,media_anual,tasa_anual,contribucion
str,str,i64,i64,i64,f64,f64,f64,f64
"""2002–2008""","""Nacidos en España""",38701173,39790019,1088846,2.81,181474.33,0.46,23.5
"""2002–2008""","""Nacidos en el extranjero""",2334098,5878919,3544821,151.87,590803.5,16.64,76.5
"""2008–2014""","""Nacidos en España""",39790019,40542974,752955,1.89,125492.5,0.31,91.07
"""2008–2014""","""Nacidos en el extranjero""",5878919,5952770,73851,1.26,12308.5,0.21,8.93
"""2014–2019""","""Nacidos en España""",40542974,40369642,-173332,-0.43,-34666.4,-0.09,-40.96
"""2014–2019""","""Nacidos en el extranjero""",5952770,6549309,596539,10.02,119307.8,1.93,140.96
"""2019–2025""","""Nacidos en España""",40369642,39664087,-705555,-1.75,-117592.5,-0.29,-31.94
"""2019–2025""","""Nacidos en el extranjero""",6549309,9464210,2914901,44.51,485816.83,6.33,131.94
"""2002–2025""","""Nacidos en España""",38701173,39664087,962914,2.49,41865.83,0.11,11.9


In [12]:
grafico_anual(nacim, total, "Variación anual por país de nacimiento", COL_NACIM, LIMITES).show()
grafico_periodos(res_nacim, COL_NACIM, "Variación media anual por periodo y país de nacimiento (% = contribución)").show()
grafico_cuota(nacim, total, COL_NACIM, "Peso de la población nacida en el extranjero", excluir=["Nacidos en España"]).show()

**Lectura 5.2 · País de nacimiento** · *borrador*

- La población nacida en el extranjero pasa de 2.334.098 (5,7 %) a 9.464.210 (19,3 %): +7.130.112, el **88,1 %** del crecimiento total. Los nacidos en España suman solo +962.914 (11,9 %).
- Por periodos: 76,5 % (2002–2008), 8,9 % (2008–2014), **141,0 %** (2014–2019) y **131,9 %** (2019–2025). Contribuciones por encima de 100 % significan que el resto de la población **cae**: los nacidos en España bajan −173.332 en 2014–2019 y −705.555 en 2019–2025.
- Con esta definición, el total (88,1 %) apunta a A con confianza Alta, y 3 de los 4 periodos apuntan a A (2002–2008 con confianza Media, 76,5 %); 2008–2014 apunta a B. La versión literal («solo», ≥ 95 %) no se cumple en el total, pero sí en 2014–2019 y 2019–2025.
- Cuidado con el significado: «nacido en el extranjero» incluye a españoles nacidos fuera (nacionalizados u otros; esta tabla no permite distinguirlos) y a extranjeros nacidos fuera; no equivale a «llegada reciente» (ver 5.3).

### 5.3 Cruce nacionalidad × país de nacimiento (cuatro grupos)

In [13]:
cuatro = grupos(base, "cuatro")
res_4 = resumen_periodos(cuatro, LIMITES)
display(*comprobar(cuatro, total))
tabla(res_4)

grupo,n,desde,hasta
str,u32,i32,i32
"""Españoles nacidos en España""",24,2002,2025
"""Españoles nacidos en el extran…",24,2002,2025
"""Extranjeros nacidos en España""",24,2002,2025
"""Extranjeros nacidos en el extr…",24,2002,2025


dif_max
i64
1


periodo,grupo,pob_ini,pob_fin,var_abs,var_pct_total,media_anual,tasa_anual,contribucion
str,str,i64,i64,i64,f64,f64,f64,f64
"""2002–2008""","""Españoles nacidos en España""",38621928,39537144,915216,2.37,152536.0,0.39,19.75
"""2002–2008""","""Españoles nacidos en el extran…",675371,1045500,370129,54.8,61688.17,7.55,7.99
"""2002–2008""","""Extranjeros nacidos en España""",79245,252875,173630,219.11,28938.33,21.34,3.75
"""2002–2008""","""Extranjeros nacidos en el extr…",1658727,4833420,3174693,191.39,529115.5,19.51,68.51
"""2008–2014""","""Españoles nacidos en España""",39537144,40109398,572254,1.45,95375.67,0.24,69.21
…,…,…,…,…,…,…,…,…
"""2019–2025""","""Extranjeros nacidos en el extr…",4334178,6412828,2078650,47.96,346441.67,6.75,94.08
"""2002–2025""","""Españoles nacidos en España""",38621928,39164944,543016,1.41,23609.39,0.06,6.71
"""2002–2025""","""Españoles nacidos en el extran…",675371,3051382,2376011,351.81,103304.83,6.78,29.36


In [14]:
grafico_anual(cuatro, total, "Variación anual por nacionalidad y país de nacimiento", COL_4, LIMITES).show()
grafico_periodos(res_4, COL_4, "Variación media anual por periodo (% = contribución)").show()
grafico_cuota(cuatro, total, COL_4, "Peso de cada grupo en la población total", excluir=["Españoles nacidos en España"]).show()

**Lectura 5.3 · Cuatro grupos** · *borrador*

- **Extranjeros nacidos en el extranjero** (el grupo más cercano a «inmigrante recién llegado»): +4.754.101 en 2002–2025, el 58,7 % del crecimiento. Sus aportes son muy desiguales: +3.174.693 (2002–2008), −590.644 (2008–2014), +91.402 (2014–2019) y +2.078.650 (2019–2025, 94,1 % del crecimiento del periodo).
- **Españoles nacidos en el extranjero:** de 675.371 a 3.051.382 (+2.376.011, 29,4 % del total). Crecen en los cuatro periodos, también cuando la población extranjera cae; es la pieza que explica gran parte de la diferencia entre las dos definiciones de 5.1 y 5.2.
- **Españoles nacidos en España:** solo +543.016 en 23 años (6,7 %), y con **descensos** en 2014–2019 (−256.340) y 2019–2025 (−688.114).
- **Extranjeros nacidos en España:** de 79.245 a 499.143 (+5,2 % del crecimiento), un grupo pequeño pero con tasas anuales altas.
- Este cruce muestra que ni «extranjero» ni «nacido fuera» son una sola cosa: con la nacionalidad se cuentan como extranjeros 499.143 personas nacidas aquí, y con el nacimiento se cuentan como inmigrantes 3,05 M de personas con nacionalidad española.

### 5.4 Las dos definiciones de «inmigrante» a la vez

In [15]:
ext = pl.concat([nac.filter(pl.col("grupo") == "Extranjera"), nacim.filter(pl.col("grupo") == "Nacidos en el extranjero")])
grafico_cuota(ext, total, {"Extranjera": ACCENT, "Nacidos en el extranjero": PALETTE[1]},
              "Nacionalidad extranjera vs. nacidos en el extranjero (% de la población)").show()

**Lectura 5.4 · Las dos definiciones** · *borrador*

- Ambos pesos suben, pero a distinto nivel: nacionalidad extranjera 4,2 % → 14,1 %; nacidos en el extranjero 5,7 % → 19,3 %.
- La brecha entre ambas definiciones pasa de unos 0,6 M de personas (1,5 puntos) en 2002 a unos 2,6 M (5,2 puntos) en 2025. Un artículo que diga «el X % son inmigrantes» debe aclarar qué definición usa, porque la diferencia es de cinco puntos.

### 5.5 Etapas de gobierno (subpregunta)

Comparar el crecimiento de la población extranjera durante el gobierno de Sánchez con el de las etapas previas. Los stocks son a 1 de enero, así que el inicio de Sánchez se aproxima con 2018 (moción de censura, junio de 2018) o 2020 (gobierno de coalición); los demás límites aproximan los cambios de gobierno de 2004 y 2011. El último dato disponible es 1-I-2025.

In [16]:
for nombre, lim in LIM_GOB.items():
    print(nombre)
    for g, a in [("Extranjera", nac), ("Nacidos en el extranjero", nacim)]:
        display(tabla(resumen_periodos(a, lim, con_total=False).filter(pl.col("grupo") == g)))

Sánchez desde 2018


periodo,grupo,pob_ini,pob_fin,var_abs,var_pct_total,media_anual,tasa_anual,contribucion
str,str,i64,i64,i64,f64,f64,f64,f64
"""2002–2004""","""Extranjera""",1737972,2900788,1162816,66.91,581408.0,29.19,76.9
"""2004–2012""","""Extranjera""",2900788,5236030,2335242,80.5,291905.25,7.66,54.68
"""2012–2018""","""Extranjera""",5236030,4577322,-658708,-12.58,-109784.67,-2.22,380.44
"""2018–2025""","""Extranjera""",4577322,6911971,2334649,51.0,333521.29,6.06,94.02


periodo,grupo,pob_ini,pob_fin,var_abs,var_pct_total,media_anual,tasa_anual,contribucion
str,str,i64,i64,i64,f64,f64,f64,f64
"""2002–2004""","""Nacidos en el extranjero""",2334098,3547669,1213571,51.99,606785.5,23.29,80.25
"""2004–2012""","""Nacidos en el extranjero""",3547669,6294952,2747283,77.44,343410.38,7.43,64.33
"""2012–2018""","""Nacidos en el extranjero""",6294952,6207509,-87443,-1.39,-14573.83,-0.23,50.5
"""2018–2025""","""Nacidos en el extranjero""",6207509,9464210,3256701,52.46,465243.0,6.21,131.15


Sánchez desde 2020


periodo,grupo,pob_ini,pob_fin,var_abs,var_pct_total,media_anual,tasa_anual,contribucion
str,str,i64,i64,i64,f64,f64,f64,f64
"""2002–2004""","""Extranjera""",1737972,2900788,1162816,66.91,581408.0,29.19,76.9
"""2004–2012""","""Extranjera""",2900788,5236030,2335242,80.5,291905.25,7.66,54.68
"""2012–2020""","""Extranjera""",5236030,5241278,5248,0.1,656.0,0.01,1.05
"""2020–2025""","""Extranjera""",5241278,6911971,1670693,31.88,334138.6,5.69,92.29


periodo,grupo,pob_ini,pob_fin,var_abs,var_pct_total,media_anual,tasa_anual,contribucion
str,str,i64,i64,i64,f64,f64,f64,f64
"""2002–2004""","""Nacidos en el extranjero""",2334098,3547669,1213571,51.99,606785.5,23.29,80.25
"""2004–2012""","""Nacidos en el extranjero""",3547669,6294952,2747283,77.44,343410.38,7.43,64.33
"""2012–2020""","""Nacidos en el extranjero""",6294952,7014753,719801,11.43,89975.12,1.36,144.01
"""2020–2025""","""Nacidos en el extranjero""",7014753,9464210,2449457,34.92,489891.4,6.17,135.31


**Lectura 5.5 · Etapas de gobierno** · *borrador*

Datos hasta 1-I-2025; los periodos de gobierno se aproximan con stocks a 1 de enero.

- **Desde 2018** (hasta 1-I-2025): nacionalidad extranjera +2.334.649 (+51 %; 333.521/año; 94,0 % del crecimiento total del periodo); nacidos en el extranjero +3.256.701 (465.243/año).
- **Desde 2020:** nacionalidad extranjera +1.670.693 (334.139/año); nacidos en el extranjero +2.449.457 (489.891/año).
- **Etapas previas** (nacionalidad extranjera): 2002–2004, 581.408/año; 2004–2012, 291.905/año; 2012–2018, **−109.785/año**. Con inicio en 2020, 2012–2020 queda plano (+0,1 %), mientras los nacidos en el extranjero crecen +11,4 %: la diferencia apunta a nacionalizaciones.
- **Sobre los «6 millones».** Medido como **variación del stock** hasta 1-I-2025, no se llega a esa cifra con ninguna de las dos definiciones y ninguno de los dos inicios (máximo: +3,26 M nacidos en el extranjero desde 2018). Pero un stock no es un flujo: las llegadas brutas (que incluyen retornos y no descuentan salidas) pueden ser mayores; eso se evalúa en Q4. Tampoco hay datos todavía de 2026.
- Ritmo comparable: el de 2002–2004 (581.408/año) es más alto que el actual (~334.000/año); el actual es más alto que el de 2004–2012 (291.905/año). La población extranjera es hoy mayor, así que en porcentaje sí es menor.

### 5.6 Veredicto Q2 (según los criterios de la sección 0)

In [17]:
t_nac = tabla_veredictos(res_nac, "Extranjera")
t_nacim = tabla_veredictos(res_nacim, "Nacidos en el extranjero")
g_nac, g_nacim = veredicto_global(t_nac), veredicto_global(t_nacim)
final = g_nac if g_nac == g_nacim else "Mixta"
display(t_nac, t_nacim)
print("Nacionalidad     →", frase(t_nac, g_nac))
print("País de nacimiento →", frase(t_nacim, g_nacim))
print("Veredicto Q2 (definiciones " + ("coinciden" if g_nac == g_nacim else "discrepan; confianza ≤ Media") + "):", final)

periodo,tipo,contribucion,literal,veredicto,confianza
str,str,f64,bool,str,str
"""2002–2008""","""periodo""",72.3,false,"""Mixta""","""Media"""
"""2008–2014""","""periodo""",-49.6,false,"""Apunta a B""","""Alta"""
"""2014–2019""","""periodo""",41.2,false,"""Mixta""","""Alta"""
"""2019–2025""","""periodo""",93.3,false,"""Apunta a A""","""Alta"""
"""2002–2025""","""total""",63.9,false,"""Mixta""","""Alta"""


periodo,tipo,contribucion,literal,veredicto,confianza
str,str,f64,bool,str,str
"""2002–2008""","""periodo""",76.5,false,"""Apunta a A""","""Media"""
"""2008–2014""","""periodo""",8.9,false,"""Apunta a B""","""Alta"""
"""2014–2019""","""periodo""",141.0,true,"""Apunta a A""","""Alta"""
"""2019–2025""","""periodo""",131.9,true,"""Apunta a A""","""Alta"""
"""2002–2025""","""total""",88.1,false,"""Apunta a A""","""Alta"""


Nacionalidad     → Mixta: la inmigración explica el 63,9 %. La versión literal («solo») no se cumple.
País de nacimiento → Apunta a A: la inmigración explica el 88,1 %. La versión literal («solo») no se cumple.
Veredicto Q2 (definiciones discrepan; confianza ≤ Media): Mixta


**Lectura 5.6 · Veredicto Q2** · *borrador (aplica los criterios pre-registrados sin cambiarlos)*

- **Nacionalidad:** *Mixta*. «La inmigración explica el 63,9 %. La versión literal («solo») no se cumple.»
- **País de nacimiento:** *Apunta a A*. «La inmigración explica el 88,1 %. La versión literal («solo») no se cumple.»
- **Veredicto Q2:** como las dos definiciones no coinciden, **Mixta con confianza ≤ Media**.
- En ambas definiciones, el patrón por periodos es el mismo: la inmigración explica casi todo el crecimiento en 2019–2025, una parte alta en 2002–2008, y poco o nada en 2008–2014 (cuando la población extranjera retrocede); los nacidos en España **decrecen** desde 2014. Pero el total de 23 años mezcla etapas muy distintas.
- **Lo que no dice este análisis:** no mide cuántos han llegado ni por qué crece la población (Q3 y Q4), y la clasificación depende de la definición de «inmigrante» (por ejemplo, 2,4 M de españoles nacidos fuera).

## 6. Q3 · ¿Cuánto del cambio de población se explica por saldo natural y por saldo migratorio exterior?

> **PENDIENTE DE DATOS** (los descargas y procesas tú).

**Datos necesarios**
- Movimiento Natural de la Población (MNP): nacimientos y defunciones anuales (total nacional; ideal: nacimientos por nacionalidad de la madre).
- Estadística de Migraciones / Estadística de Variaciones Residenciales (EMCR): inmigraciones y emigraciones con el exterior, anuales, total y por nacionalidad.

**Cálculo previsto** (por año natural y por periodo)
- `saldo_natural = nacimientos − defunciones`; `saldo_migratorio = inmigraciones − emigraciones`.
- `residuo = ΔPoblación − saldo_natural − saldo_migratorio` (se muestra siempre; si |residuo| > 20 % de ΔPoblación → *Insuficiente*).
- `contribucion_migratoria = saldo_migratorio / ΔPoblación` (puede superar 100 %), con los mismos umbrales A/B/Mixta de la sección 0.
- Los flujos son de año natural y el stock a 1 de enero: ΔPoblación(t) = stock(1-I-t+1) − stock(1-I-t) debe compararse con los flujos del año t.

**Cautela:** los nacidos de madre extranjera aparecen en el saldo natural aunque su origen sea migratorio.

In [18]:
# TODO (Q3): cargar MNP y EMCR -> tabla anual con columnas año, nacimientos, defunciones, inmigraciones, emigraciones
# y reutilizar variacion_anual / resumen_periodos / clasificar.

## 7. Q4 · ¿Cuántos llegan, cuántos salen y cuántos se quedan? (la cifra de «6 millones»)

> **PENDIENTE DE DATOS** (los descargas y procesas tú).

**Datos necesarios:** EMCR (inmigraciones y emigraciones con el exterior por año, total y por nacionalidad), y el stock de nacidos en el extranjero / extranjeros a 1 de enero (ya disponible arriba hasta 2025, y 2026 desde las tablas 60129/60130).

**Medidas** (desde 2018 y desde 2020 hasta el último dato): llegadas brutas (todas / extranjeros), saldo neto (todos / extranjeros) y variación del stock de nacidos en el extranjero.

**Regla de decisión:** *Respaldada* si el saldo neto está entre 5,4 y 6,6 M; *Depende de la medida* si solo las llegadas brutas llegan a esa cifra; *No respaldada* si ninguna alcanza 5,4 M. Coherencia flujo–stock: se señala si |Δstock − saldo de extranjeros| > 25 % de Δstock.

**Cautela:** las llegadas brutas incluyen españoles que regresan y no descuentan las salidas.

In [19]:
# TODO (Q4): flujos EMCR por año y nacionalidad; comparar saldo neto de extranjeros con Δ del stock (sección 5.1).

## 8. Notas y limitaciones

- **Stock ≠ flujo.** Los aumentos de población extranjera son variaciones de stock: incluyen llegadas, salidas, nacimientos, defunciones, cambios de nacionalidad y revisiones del padrón. Q4 los relaciona con flujos.
- **Dos definiciones de «inmigrante»** (nacionalidad y país de nacimiento) no coinciden: las adquisiciones de nacionalidad española pasan población de «extranjera» a «española» sin que nadie se mueva.
- **Datos hasta 1-I-2025** en esta tabla. Para 2026 hay que empalmar con las tablas 60129/60130 tras comprobar la coherencia en el solape (posible ruptura de serie).
- **Alcance nacional.** Los resultados territoriales (CCAA, provincias) irán en un artículo aparte; el trabajo previo está en `i01-population-nationality.backup-2026-10-07.ipynb`.

## 9. Registro de cambios de los criterios

- 2026-10-07: tesis A rebajada de «todo» (≥ 95 %) a «la mayor parte» (≥ 75 %), B simetrizada (< 25 %, antes < 50 %); motivo: versión anterior asimétrica; la versión literal se mantiene como etiqueta.